In [18]:
!pip install stable-baselines3[extra]
!pip install gymnasium[atari,accept-rom-license]
!pip install ale-py

In [24]:
%%writefile train.py
from stable_baselines3.common.callbacks import (
    EvalCallback, CheckpointCallback, CallbackList,)
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.vec_env import VecFrameStack
from stable_baselines3.common.env_util import make_atari_env
from stable_baselines3 import DQN
import gymnasium as gym
import ale_py
import argparse
import json
import os

# % % writefile train.py


# Argument Parser
def parse_args():
    parser = argparse.ArgumentParser(
        description="Train a DQN agent on an Atari environment.")

    parser.add_argument("--experiment-name", type=str, default="baseline")
    parser.add_argument("--env", type=str, default="ALE/Pong-v5")
    parser.add_argument("--policy", type=str, default="CnnPolicy",
                        choices=["CnnPolicy", "MlpPolicy"])

    parser.add_argument("--timesteps", type=int, default=500_000)
    parser.add_argument("--seed", type=int, default=42)

    parser.add_argument("--lr", type=float, default=1e-4)
    parser.add_argument("--gamma", type=float, default=0.99)
    parser.add_argument("--batch-size", type=int, default=32)
    parser.add_argument("--buffer-size", type=int, default=100_000)
    parser.add_argument("--learning-starts", type=int, default=20_000)

    parser.add_argument("--train-freq", type=int, default=4)
    parser.add_argument("--target-update", type=int, default=10_000)

    parser.add_argument("--eps-start", type=float, default=1.0)
    parser.add_argument("--eps-end", type=float, default=0.05)
    parser.add_argument("--eps-fraction", type=float, default=0.10)

    parser.add_argument("--eval-freq", type=int, default=10_000)
    parser.add_argument("--checkpoint-freq", type=int, default=100_000)

    parser.add_argument("--verbose", type=int, default=0)

    return parser.parse_args()


# Environment Setup
def create_env(env_name, seed, policy):
    if policy == "CnnPolicy":
        env = make_atari_env(env_name, n_envs=1, seed=seed)
        env = VecFrameStack(env, n_stack=4)
    else:
        env = make_vec_env(
            env_name,
            n_envs=1,
            seed=seed,
            env_kwargs={"obs_type": "ram"}
        )

    return env


# DQN Model
def create_model(env, args, log_dir):

    model = DQN(
        policy=args.policy,
        env=env,

        learning_rate=args.lr,
        gamma=args.gamma,

        batch_size=args.batch_size,
        buffer_size=args.buffer_size,
        learning_starts=args.learning_starts,

        train_freq=args.train_freq,
        target_update_interval=args.target_update,

        exploration_initial_eps=args.eps_start,
        exploration_final_eps=args.eps_end,
        exploration_fraction=args.eps_fraction,

        verbose=args.verbose,

        tensorboard_log=log_dir,
    )

    return model


# Custom Evaluation Callback
class CompactEvalCallback(EvalCallback):
    def __init__(self, *args, csv_file=None, **kwargs):
        super().__init__(*args, **kwargs)

        self.csv_file = csv_file

        # Create CSV and write header
        if self.csv_file is not None:
            with open(self.csv_file, "w") as f:
                f.write("timesteps,reward,best_reward,exploration_rate\n")

    def _on_step(self) -> bool:
        continue_training = super()._on_step()

        if self.eval_freq > 0 and self.n_calls % self.eval_freq == 0:

            print(
                f"[{self.num_timesteps:>7,}] "
                f"Reward: {self.last_mean_reward:>7.2f} | "
                f"Best: {self.best_mean_reward:>7.2f} | "
                f"Exploration: {self.model.exploration_rate:.3f}"
            )

            # Save results to CSV
            if self.csv_file is not None:
                with open(self.csv_file, "a") as f:
                    f.write(
                        f"{self.num_timesteps},"
                        f"{self.last_mean_reward},"
                        f"{self.best_mean_reward},"
                        f"{self.model.exploration_rate}\n"
                    )

        return continue_training

# Training


def train(args):
    base_dir = "/kaggle/working/"

    # assert os.path.isdir("/content/drive/MyDrive"), (
    #     "Google Drive doesn't appear to be mounted. "
    #     "Run drive.mount('/content/drive') in a cell before training."
    # )
    model_dir = os.path.join(base_dir, "models", args.experiment_name)
    log_dir = os.path.join(base_dir, "logs", args.experiment_name, )

    checkpoint_dir = os.path.join(
        base_dir, "checkpoints", args.experiment_name, )

    os.makedirs(model_dir, exist_ok=True)
    os.makedirs(log_dir, exist_ok=True)
    os.makedirs(checkpoint_dir, exist_ok=True)

    with open(
        os.path.join(model_dir, "config.json"),
        "w",
    ) as f:

        json.dump(
            vars(args),
            f,
            indent=4,
        )

    env = create_env(args.env, args.seed, args.policy, )

    eval_env = create_env(args.env, args.seed + 100, args.policy, )

    model = create_model(env, args, log_dir,)

    checkpoint_callback = CheckpointCallback(
        save_freq=args.checkpoint_freq,
        save_path=checkpoint_dir,
        name_prefix="dqn_checkpoint",
    )

    eval_callback = CompactEvalCallback(
        eval_env,
        best_model_save_path=model_dir,
        log_path=log_dir,
        eval_freq=args.eval_freq,
        deterministic=True,
        render=False,
        verbose=0,
        n_eval_episodes=10,
    )

    callbacks = CallbackList([checkpoint_callback, eval_callback,])

    try:
        model.learn(
            total_timesteps=args.timesteps,
            callback=callbacks,
            log_interval=10,
        )

        model.save(os.path.join(model_dir, "dqn_model", ))

        print("\nTraining completed successfully!")
        print(f"Final model    : {os.path.join(model_dir, 'dqn_model.zip')}")
        print(f"Best model     : {os.path.join(model_dir, 'best_model.zip')}")
        print(f"Config         : {os.path.join(model_dir, 'config.json')}")

    finally:
        env.close()
        eval_env.close()


# Main

def main():
    args = parse_args()
    train(args)


if __name__ == "__main__":
    main()

Overwriting train.py


# Experiment 11 
gamma=0.90

Starting the sweep at the low end to see how much a short-horizon agent suffers on
Pong

In [27]:
!python train.py --experiment-name sheryl_exp11 --policy CnnPolicy --timesteps 200000 --seed 42 --lr 1e-4 --gamma 0.90 --batch-size 32 --buffer-size 100000 --learning-starts 20000 --train-freq 4 --target-update 10000 --eps-start 1.0 --eps-end 0.05 --eps-fraction 0.10

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
A.L.E: Arcade Learning Environment (version 0.11.2+ecc1138)
[Powered by Stella]
/usr/local/lib/python3.12/dist-packages/stable_baselines3/common/callbacks.py:419: UserWarning: Training and eval env are not of the same type<stable_baselines3.common.vec_env.vec_transpose.VecTransposeImage object at 0x7ac35bf2f7a0> != <stable_baselines3.common.vec_env.vec_frame_stack.VecFrameStack object at 0x7ac35bf5cd10>
  warnings.warn("Training and eval env are not of the same type" f"{self.training_env} != {se

In [ ]:
Final reward -16.00, best -15.10 which is a reasonable early result for
such a low gamma.

Gamma this low doesn't seem to hurt much. Move upward in
small steps to map whether performance keeps improving as gamma increases,
or is already close to its ceiling.

# Experiment 12 
gamma=0.92

In [28]:
!python train.py --experiment-name sheryl_exp12 --policy CnnPolicy --timesteps 200000 --seed 42 --lr 1e-4 --gamma 0.92 --batch-size 32 --buffer-size 100000 --learning-starts 20000 --train-freq 4 --target-update 10000 --eps-start 1.0 --eps-end 0.05 --eps-fraction 0.10

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
A.L.E: Arcade Learning Environment (version 0.11.2+ecc1138)
[Powered by Stella]
/usr/local/lib/python3.12/dist-packages/stable_baselines3/common/callbacks.py:419: UserWarning: Training and eval env are not of the same type<stable_baselines3.common.vec_env.vec_transpose.VecTransposeImage object at 0x7818de385b20> != <stable_baselines3.common.vec_env.vec_frame_stack.VecFrameStack object at 0x7818de386960>
  warnings.warn("Training and eval env are not of the same type" f"{self.training_env} != {se

Final reward -17.50, best -17.50, worse than Experiment 11,
the weakest result in the low-gamma region so far.

Unclear if this is a real effect of gamma or run-to-run
noise (single seed per configuration). Continue upward to see if the dip
is part of a trend or an outlier.

# Experiment 13 
gamma=0.94

In [29]:
!python train.py --experiment-name sheryl_exp13 --policy CnnPolicy --timesteps 200000 --seed 42 --lr 1e-4 --gamma 0.94 --batch-size 32 --buffer-size 100000 --learning-starts 20000 --train-freq 4 --target-update 10000 --eps-start 1.0 --eps-end 0.05 --eps-fraction 0.10

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
A.L.E: Arcade Learning Environment (version 0.11.2+ecc1138)
[Powered by Stella]
/usr/local/lib/python3.12/dist-packages/stable_baselines3/common/callbacks.py:419: UserWarning: Training and eval env are not of the same type<stable_baselines3.common.vec_env.vec_transpose.VecTransposeImage object at 0x7c4fb9976000> != <stable_baselines3.common.vec_env.vec_frame_stack.VecFrameStack object at 0x7c4fbbb45940>
  warnings.warn("Training and eval env are not of the same type" f"{self.training_env} != {se

Final reward -16.60, best -15.10, recovered to match
Experiment 11's best score.

This confirms Experiment 12's dip was likely noise rather
than a real gamma effect. Continue increasing gamma to check for a
smooth trend.

# Experiment 14 
gamma=0.96

In [30]:
!python train.py --experiment-name sheryl_exp14 --policy CnnPolicy --timesteps 200000 --seed 42 --lr 1e-4 --gamma 0.96 --batch-size 32 --buffer-size 100000 --learning-starts 20000 --train-freq 4 --target-update 10000 --eps-start 1.0 --eps-end 0.05 --eps-fraction 0.10

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
A.L.E: Arcade Learning Environment (version 0.11.2+ecc1138)
[Powered by Stella]
/usr/local/lib/python3.12/dist-packages/stable_baselines3/common/callbacks.py:419: UserWarning: Training and eval env are not of the same type<stable_baselines3.common.vec_env.vec_transpose.VecTransposeImage object at 0x7c89a4237ce0> != <stable_baselines3.common.vec_env.vec_frame_stack.VecFrameStack object at 0x7c89a4234cb0>
  warnings.warn("Training and eval env are not of the same type" f"{self.training_env} != {se

Final reward -16.00, best -16.00 — steady, in line with the
other mid-range gammas tested so far.

There is no clear improvement yet. Push closer to the standard 0.99
region (testing 0.97) to see whether results keep climbing as gamma
increases, or plateau before reaching it.

# Experiment 15 
gamma=0.97

In [31]:
!python train.py --experiment-name sheryl_exp15 --policy CnnPolicy --timesteps 200000 --seed 42 --lr 1e-4 --gamma 0.97 --batch-size 32 --buffer-size 100000 --learning-starts 20000 --train-freq 4 --target-update 10000 --eps-start 1.0 --eps-end 0.05 --eps-fraction 0.10

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
A.L.E: Arcade Learning Environment (version 0.11.2+ecc1138)
[Powered by Stella]
/usr/local/lib/python3.12/dist-packages/stable_baselines3/common/callbacks.py:419: UserWarning: Training and eval env are not of the same type<stable_baselines3.common.vec_env.vec_transpose.VecTransposeImage object at 0x78a5f2b75f10> != <stable_baselines3.common.vec_env.vec_frame_stack.VecFrameStack object at 0x78a5f2d866c0>
  warnings.warn("Training and eval env are not of the same type" f"{self.training_env} != {se

Final reward -14.90, best -14.90, the best result of the
entire sweep by a clear margin.

This is the strongest configuration found so far. Test just
above 0.97 to check whether this is a local peak or part of a continued
upward trend as gamma approaches 1.

# Experiment 16 
gamma=0.98

In [38]:
!python train.py --experiment-name sheryl_exp16 --policy CnnPolicy --timesteps 200000 --seed 42 --lr 1e-4 --gamma 0.98 --batch-size 32 --buffer-size 100000 --learning-starts 20000 --train-freq 4 --target-update 10000 --eps-start 1.0 --eps-end 0.05 --eps-fraction 0.10

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
A.L.E: Arcade Learning Environment (version 0.11.2+ecc1138)
[Powered by Stella]
/usr/local/lib/python3.12/dist-packages/stable_baselines3/common/callbacks.py:419: UserWarning: Training and eval env are not of the same type<stable_baselines3.common.vec_env.vec_transpose.VecTransposeImage object at 0x7ade45869e20> != <stable_baselines3.common.vec_env.vec_frame_stack.VecFrameStack object at 0x7ade45db66c0>
  warnings.warn("Training and eval env are not of the same type" f"{self.training_env} != {se

 Final reward -16.20, best -15.40 — worse than 0.97 despite
being the very next step up.

This suggests 0.97 is a local peak rather than the start of
a continued upward trend. Shift focus to gamma values very close to 1, to
see how the agent behaves at the extreme end of the range.

# Experiment 17 
gamma=0.995

**Why:** First test deep into near-1 territory, to see whether pushing
gamma much higher continues to hurt performance or stabilizes somewhere.

In [39]:
!python train.py --experiment-name sheryl_exp17 --policy CnnPolicy --timesteps 200000 --seed 42 --lr 1e-4 --gamma 0.995 --batch-size 32 --buffer-size 100000 --learning-starts 20000 --train-freq 4 --target-update 10000 --eps-start 1.0 --eps-end 0.05 --eps-fraction 0.10

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
A.L.E: Arcade Learning Environment (version 0.11.2+ecc1138)
[Powered by Stella]
/usr/local/lib/python3.12/dist-packages/stable_baselines3/common/callbacks.py:419: UserWarning: Training and eval env are not of the same type<stable_baselines3.common.vec_env.vec_transpose.VecTransposeImage object at 0x7d164ec55e50> != <stable_baselines3.common.vec_env.vec_frame_stack.VecFrameStack object at 0x7d164edb4e60>
  warnings.warn("Training and eval env are not of the same type" f"{self.training_env} != {se

Final reward -18.20, best -18.20, a sharp drop-off from every other configuration tested.

A gamma this close to 1 asks the Q-function to propagate
credit across very long horizons, which plausibly needs more than 200k
steps to stabilize. Test the next value up to see if this is a hard cliff
or a gradual decline.

# Experiment 18 
gamma=0.997

In [40]:
!python train.py --experiment-name sheryl_exp18 --policy CnnPolicy --timesteps 200000 --seed 42 --lr 1e-4 --gamma 0.997 --batch-size 32 --buffer-size 100000 --learning-starts 20000 --train-freq 4 --target-update 10000 --eps-start 1.0 --eps-end 0.05 --eps-fraction 0.10

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
A.L.E: Arcade Learning Environment (version 0.11.2+ecc1138)
[Powered by Stella]
/usr/local/lib/python3.12/dist-packages/stable_baselines3/common/callbacks.py:419: UserWarning: Training and eval env are not of the same type<stable_baselines3.common.vec_env.vec_transpose.VecTransposeImage object at 0x7811f5d1d910> != <stable_baselines3.common.vec_env.vec_frame_stack.VecFrameStack object at 0x7811f5c4ec30>
  warnings.warn("Training and eval env are not of the same type" f"{self.training_env} != {se

Final reward -17.60, best -15.90, recovered somewhat from
0.995, though still one of the weaker results overall.

**Decision:** Not a total collapse (unlike what a runaway learning-rate
failure would look like), but consistently underperforming the low/mid
gamma region. Continue upward to map the rest of the high-gamma region
before drawing a conclusion.

# Experiment 19 
gamma=0.999

In [41]:
!python train.py --experiment-name sheryl_exp19 --policy CnnPolicy --timesteps 200000 --seed 42 --lr 1e-4 --gamma 0.999 --batch-size 32 --buffer-size 100000 --learning-starts 20000 --train-freq 4 --target-update 10000 --eps-start 1.0 --eps-end 0.05 --eps-fraction 0.10

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
A.L.E: Arcade Learning Environment (version 0.11.2+ecc1138)
[Powered by Stella]
/usr/local/lib/python3.12/dist-packages/stable_baselines3/common/callbacks.py:419: UserWarning: Training and eval env are not of the same type<stable_baselines3.common.vec_env.vec_transpose.VecTransposeImage object at 0x7f3b901c1d30> != <stable_baselines3.common.vec_env.vec_frame_stack.VecFrameStack object at 0x7f3b9098b980>
  warnings.warn("Training and eval env are not of the same type" f"{self.training_env} != {se

**Result:** Final reward -17.20, best -17.00 essentially unchanged from
0.997, no further degradation.

**Decision:** Performance appears to plateau in the high-gamma region
rather than continuing to worsen. Test the most extreme value (0.9995) to
close out the sweep and confirm this plateau holds all the way to the
edge of the range.

# Experiment 20 
gamma=0.9995

**Why:** Final upper-bound test, nearly indistinguishable from gamma=1,
to close out the sweep on the high end.

In [42]:
!python train.py --experiment-name sheryl_exp20 --policy CnnPolicy --timesteps 200000 --seed 42 --lr 1e-4 --gamma 0.9995 --batch-size 32 --buffer-size 100000 --learning-starts 20000 --train-freq 4 --target-update 10000 --eps-start 1.0 --eps-end 0.05 --eps-fraction 0.10

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
A.L.E: Arcade Learning Environment (version 0.11.2+ecc1138)
[Powered by Stella]
/usr/local/lib/python3.12/dist-packages/stable_baselines3/common/callbacks.py:419: UserWarning: Training and eval env are not of the same type<stable_baselines3.common.vec_env.vec_transpose.VecTransposeImage object at 0x7d2360b6e060> != <stable_baselines3.common.vec_env.vec_frame_stack.VecFrameStack object at 0x7d23610b7a40>
  warnings.warn("Training and eval env are not of the same type" f"{self.training_env} != {se

**Result:** Final reward -16.70, best -15.60, the best-performing run in
the high-gamma group, but still well short of the 0.97 peak.

**Decision:** Closes the sweep. The viable gamma range for this training
budget is roughly 0.94–0.98, with 0.97 as the best value found.
Gamma ≥0.995 was consistently the weakest region tested, suggesting very
high discount factors need more than 200k steps to stabilize on this task.